# Whisper Speech-to-Text Inference in JAX on TPU

A from-scratch **Whisper** inference pipeline in pure JAX, built for TPU:

1. **Load OpenAI's Whisper weights** from the Hugging Face Hub (`safetensors`), any size from `tiny` to `large-v3` / `large-v3-turbo`.
2. **Log-mel spectrogram on the TPU**: STFT + mel filterbank in `jnp`, checked against the Hugging Face feature extractor.
3. **Encoder**: conv stem + transformer, with layers stacked and run through `lax.scan` for fast compilation.
4. **Decoder with a KV cache**: greedy decoding inside a single jitted `lax.while_loop`, with cross-attention K/V computed once per utterance.
5. **Language detection**, **transcribe / translate** tasks, and token suppression matching OpenAI's generation config.
6. **Data-parallel batching** across all TPU cores using `jax.sharding`.
7. **Accuracy check**: word error rate (WER) on LibriSpeech samples.
8. **Long-form audio**: split at quiet points into ≤30 s chunks and transcribe them in one batch.
9. **Benchmark**: real-time factor vs batch size.

**Why not the `whisper-jax` package?** It is built on the Flax Whisper model in `transformers`, which `transformers` v5 removed. This notebook only needs `jax`, `numpy`, `safetensors`, and `transformers` for the tokenizer and reference feature extractor (no PyTorch or Flax needed).

**Runtime:** `Runtime → Change runtime type → TPU` (v5e-1, v6e-1 or v2-8 all work).

In [1]:
%pip install -q -U "transformers>=4.40" safetensors huggingface_hub soundfile pyarrow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [2]:
import functools, json, math, os, re, time

import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file
from transformers import WhisperFeatureExtractor, WhisperTokenizer

print("JAX", jax.__version__, "| backend:", jax.default_backend(), "| devices:", jax.device_count())
for d in jax.devices():
    print("   ", d)
ON_TPU = jax.default_backend() == "tpu"

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


JAX 0.11.2 | backend: tpu | devices: 1
    TPU_0(process=0,(0,0,0,0))


## 1. Choose a model

| model | params | notes |
|---|---|---|
| `openai/whisper-tiny` / `base` / `small` | 39M / 74M / 244M | fast, good for experimenting |
| `openai/whisper-medium` | 769M | |
| `openai/whisper-large-v3` | 1.55B | best accuracy, 128 mel bins |
| `openai/whisper-large-v3-turbo` | 809M | large-v3 encoder + 4-layer decoder, ~same accuracy, much faster |
| `*.en` variants | | English-only models (no language/task tokens) |

Everything below (mel bins, vocab size, special tokens, layer counts) is read from the checkpoint's config, so any of these work unchanged.

In [3]:
MODEL_ID = "openai/whisper-large-v3-turbo" if ON_TPU else "openai/whisper-tiny"
DTYPE = jnp.bfloat16 if ON_TPU else jnp.float32     # weights + activations; LayerNorm/softmax stay fp32

config = json.load(open(hf_hub_download(MODEL_ID, "config.json")))
gen_cfg = json.load(open(hf_hub_download(MODEL_ID, "generation_config.json")))
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID)

print(MODEL_ID)
print({k: config[k] for k in ["d_model", "encoder_layers", "decoder_layers", "encoder_attention_heads",
                              "num_mel_bins", "vocab_size", "max_source_positions", "max_target_positions"]})

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.71M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

openai/whisper-large-v3-turbo
{'d_model': 1280, 'encoder_layers': 32, 'decoder_layers': 4, 'encoder_attention_heads': 20, 'num_mel_bins': 128, 'vocab_size': 51866, 'max_source_positions': 1500, 'max_target_positions': 448}


## 2. Load the weights into a JAX-friendly pytree

The Hugging Face checkpoint stores PyTorch-layout tensors (`Linear.weight` is `[out, in]`, `Conv1d.weight` is `[out, in, k]`). We:

* transpose them to `x @ W` / `NWC` convolution layouts,
* **stack the per-layer weights along a leading axis** so the encoder and decoder become a single `lax.scan`, which compiles once regardless of depth,
* cast to bf16 and **replicate** them on every TPU core. Each core then runs a different slice of the batch.

In [4]:
def load_state_dict(model_id):
    try:   # sharded checkpoints
        index = json.load(open(hf_hub_download(model_id, "model.safetensors.index.json")))
        files = sorted(set(index["weight_map"].values()))
    except Exception:
        files = ["model.safetensors"]
    sd = {}
    for f in files:
        sd.update(load_file(hf_hub_download(model_id, f)))
    return {k.removeprefix("model."): v for k, v in sd.items()}


def convert_params(sd, cfg, dtype):
    def lin(prefix, bias=True):
        p = {"w": sd[f"{prefix}.weight"].T}
        if bias:
            p["b"] = sd[f"{prefix}.bias"]
        return p
    def ln(prefix):
        return {"scale": sd[f"{prefix}.weight"], "bias": sd[f"{prefix}.bias"]}
    def attn(prefix):
        return {"q": lin(f"{prefix}.q_proj"), "k": lin(f"{prefix}.k_proj", bias=False),
                "v": lin(f"{prefix}.v_proj"), "o": lin(f"{prefix}.out_proj")}
    def stack(trees):
        return jax.tree.map(lambda *xs: np.stack(xs), *trees)

    enc_layers = [{
        "attn_ln": ln(f"encoder.layers.{i}.self_attn_layer_norm"), "attn": attn(f"encoder.layers.{i}.self_attn"),
        "mlp_ln": ln(f"encoder.layers.{i}.final_layer_norm"),
        "fc1": lin(f"encoder.layers.{i}.fc1"), "fc2": lin(f"encoder.layers.{i}.fc2"),
    } for i in range(cfg["encoder_layers"])]
    dec_layers = [{
        "attn_ln": ln(f"decoder.layers.{i}.self_attn_layer_norm"), "attn": attn(f"decoder.layers.{i}.self_attn"),
        "xattn_ln": ln(f"decoder.layers.{i}.encoder_attn_layer_norm"), "xattn": attn(f"decoder.layers.{i}.encoder_attn"),
        "mlp_ln": ln(f"decoder.layers.{i}.final_layer_norm"),
        "fc1": lin(f"decoder.layers.{i}.fc1"), "fc2": lin(f"decoder.layers.{i}.fc2"),
    } for i in range(cfg["decoder_layers"])]

    params = {
        "encoder": {
            "conv1": {"w": sd["encoder.conv1.weight"].transpose(2, 1, 0), "b": sd["encoder.conv1.bias"]},  # [k, in, out]
            "conv2": {"w": sd["encoder.conv2.weight"].transpose(2, 1, 0), "b": sd["encoder.conv2.bias"]},
            "pos": sd["encoder.embed_positions.weight"],
            "layers": stack(enc_layers),
            "ln": ln("encoder.layer_norm"),
        },
        "decoder": {
            "embed": sd["decoder.embed_tokens.weight"],     # also the (tied) output projection
            "pos": sd["decoder.embed_positions.weight"],
            "layers": stack(dec_layers),
            "ln": ln("decoder.layer_norm"),
        },
    }
    return jax.tree.map(lambda x: jnp.asarray(x, dtype=dtype), params)


# 1-D mesh over all devices: batch is split over 'data', parameters are replicated.
mesh = Mesh(np.array(jax.devices()), ("data",))
REPLICATED = NamedSharding(mesh, P())
BATCH = NamedSharding(mesh, P("data"))

t0 = time.time()
params = jax.device_put(convert_params(load_state_dict(MODEL_ID), config, DTYPE), REPLICATED)
n = sum(x.size for x in jax.tree.leaves(params))
print(f"loaded {n/1e6:.1f}M params as {jnp.dtype(DTYPE).name} in {time.time()-t0:.1f}s")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

loaded 808.9M params as bfloat16 in 23.6s


## 3. Log-mel spectrogram on the TPU

Whisper's front end: pad/trim to 30 s (480,000 samples at 16 kHz), centered STFT (400-sample Hann window, hop 160), power spectrum, mel filterbank, `log10`, clamp to `max − 8`, then rescale. Running it under `jit` on the TPU keeps the host CPU out of the critical path.

In [5]:
SAMPLE_RATE = 16_000
N_SAMPLES = 30 * SAMPLE_RATE
N_FFT, HOP = 400, 160
N_FRAMES = N_SAMPLES // HOP          # 3000 mel frames -> 1500 encoder positions

MEL_FILTERS = jnp.asarray(feature_extractor.mel_filters, jnp.float32)            # [201, n_mels]
HANN = jnp.asarray(np.hanning(N_FFT + 1)[:-1], jnp.float32)                      # periodic Hann window

@jax.jit
def log_mel_spectrogram(audio):
    # audio: [B, 480000] float32 -> [B, 3000, n_mels]
    x = jnp.pad(audio, ((0, 0), (N_FFT // 2, N_FFT // 2)), mode="reflect")
    idx = jnp.arange(N_FRAMES + 1)[:, None] * HOP + jnp.arange(N_FFT)[None, :]
    frames = x[:, idx] * HANN                                                   # [B, 3001, 400]
    power = jnp.abs(jnp.fft.rfft(frames, n=N_FFT, axis=-1)) ** 2                 # [B, 3001, 201]
    mel = jnp.einsum("btf,fm->btm", power[:, :-1], MEL_FILTERS, precision=lax.Precision.HIGHEST)
    log_spec = jnp.log10(jnp.maximum(mel, 1e-10))
    log_spec = jnp.maximum(log_spec, log_spec.max(axis=(1, 2), keepdims=True) - 8.0)
    return (log_spec + 4.0) / 4.0


def pad_or_trim(audio, length=N_SAMPLES):
    audio = np.asarray(audio, np.float32)[:length]
    return np.pad(audio, (0, length - len(audio)))

### Load sample audio

We use 73 LibriSpeech utterances from `hf-internal-testing/librispeech_asr_dummy` (a single small parquet file) and check our features against the Hugging Face (NumPy) feature extractor.

In [6]:
import io, soundfile as sf, pyarrow.parquet as pq

table = pq.read_table(hf_hub_download("hf-internal-testing/librispeech_asr_dummy",
                                      "clean/validation-00000-of-00001.parquet", repo_type="dataset"))
samples = []
for audio, text in zip(table.column("audio").to_pylist(), table.column("text").to_pylist()):
    wav, sr = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
    assert sr == SAMPLE_RATE
    samples.append((wav, text))
total_sec = sum(len(w) for w, _ in samples) / SAMPLE_RATE
print(f"{len(samples)} utterances, {total_sec/60:.1f} min of audio")
print("example:", samples[0][1])

ours = np.asarray(log_mel_spectrogram(pad_or_trim(samples[0][0])[None]))[0].T     # [n_mels, 3000]
ref = feature_extractor(samples[0][0], sampling_rate=SAMPLE_RATE, return_tensors="np")["input_features"][0]
print(f"log-mel vs HF feature extractor: max abs diff = {np.abs(ours - ref).max():.2e}")
assert np.abs(ours - ref).max() < 1e-3

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

73 utterances, 8.0 min of audio
example: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
log-mel vs HF feature extractor: max abs diff = 1.90e-05


## 4. The model

All matmuls run in `DTYPE` (bf16 on TPU). LayerNorm, softmax and the final logits use float32.

**Encoder:** `conv(k=3) → GELU → conv(k=3, stride 2) → GELU → + positions → N × pre-LN transformer block → LayerNorm`. 30 s of audio becomes 1500 vectors.

**Decoder step** (one token for the whole batch):
* **self-attention** reads/writes a preallocated KV cache `[L, B, T_max, H, Dh]` with `dynamic_update_slice`,
* **cross-attention** uses encoder K/V that are **projected once** per utterance for all layers,
* logits come from the tied token embedding.

In [7]:
D = config["d_model"]
H = config["encoder_attention_heads"]
DH = D // H
SCALE = DH ** -0.5


def layer_norm(x, p, eps=1e-5):
    x32 = x.astype(jnp.float32)
    mu = x32.mean(-1, keepdims=True)
    var = jnp.mean((x32 - mu) ** 2, -1, keepdims=True)
    y = (x32 - mu) * lax.rsqrt(var + eps) * p["scale"].astype(jnp.float32) + p["bias"].astype(jnp.float32)
    return y.astype(x.dtype)

def dense(x, p):
    y = x @ p["w"]
    return y + p["b"] if "b" in p else y

def gelu(x):
    return jax.nn.gelu(x, approximate=False)

def split_heads(x):                    # [..., D] -> [..., H, DH]
    return x.reshape(*x.shape[:-1], H, DH)

def attention(q, k, v, mask=None):
    # q [B, Tq, H, DH], k/v [B, Tk, H, DH]
    s = jnp.einsum("bqhd,bkhd->bhqk", q, k).astype(jnp.float32)
    if mask is not None:
        s = jnp.where(mask, s, -1e30)
    w = jax.nn.softmax(s, axis=-1).astype(v.dtype)
    return jnp.einsum("bhqk,bkhd->bqhd", w, v)


# ---------------- encoder ----------------
def conv1d(x, p, stride):
    y = lax.conv_general_dilated(x, p["w"], (stride,), [(1, 1)], dimension_numbers=("NWC", "WIO", "NWC"))
    return y + p["b"]

@jax.jit
def encode(params, mel):
    # mel [B, 3000, n_mels] -> encoder states [B, 1500, D]
    p = params["encoder"]
    x = mel.astype(DTYPE)
    x = gelu(conv1d(x, p["conv1"], 1))
    x = gelu(conv1d(x, p["conv2"], 2))
    x = x + p["pos"][: x.shape[1]]

    def block(x, lp):
        h = layer_norm(x, lp["attn_ln"])
        q = split_heads(dense(h, lp["attn"]["q"]) * SCALE)
        k = split_heads(dense(h, lp["attn"]["k"]))
        v = split_heads(dense(h, lp["attn"]["v"]))
        x = x + dense(attention(q, k, v).reshape(x.shape), lp["attn"]["o"])
        h = layer_norm(x, lp["mlp_ln"])
        x = x + dense(gelu(dense(h, lp["fc1"])), lp["fc2"])
        return x, None

    x, _ = lax.scan(block, x, p["layers"])
    return layer_norm(x, p["ln"])


@jax.jit
def cross_kv(params, enc):
    # Project encoder states to cross-attention K/V for every decoder layer at once: [L, B, S, H, DH]
    xa = params["decoder"]["layers"]["xattn"]
    k = jnp.einsum("bsd,lde->lbse", enc, xa["k"]["w"])
    v = jnp.einsum("bsd,lde->lbse", enc, xa["v"]["w"]) + xa["v"]["b"][:, None, None, :]
    return split_heads(k), split_heads(v)


# ---------------- decoder (one step, KV-cached) ----------------
def decoder_step(params, token, t, cache, xkv):
    # token [B] at position t -> logits [B, V], updated cache.
    p = params["decoder"]
    x = (p["embed"][token] + p["pos"][t])[:, None, :]            # [B, 1, D]
    T_max = cache[0].shape[2]
    mask = (jnp.arange(T_max) <= t)[None, None, None, :]           # causal mask over the cache

    def block(x, inp):
        lp, ck, cv, xk, xv = inp
        h = layer_norm(x, lp["attn_ln"])
        q = split_heads(dense(h, lp["attn"]["q"]) * SCALE)
        k = split_heads(dense(h, lp["attn"]["k"]))
        v = split_heads(dense(h, lp["attn"]["v"]))
        ck = lax.dynamic_update_slice_in_dim(ck, k.astype(ck.dtype), t, axis=1)
        cv = lax.dynamic_update_slice_in_dim(cv, v.astype(cv.dtype), t, axis=1)
        x = x + dense(attention(q, ck, cv, mask).reshape(x.shape), lp["attn"]["o"])

        h = layer_norm(x, lp["xattn_ln"])
        q = split_heads(dense(h, lp["xattn"]["q"]) * SCALE)
        x = x + dense(attention(q, xk, xv).reshape(x.shape), lp["xattn"]["o"])

        h = layer_norm(x, lp["mlp_ln"])
        x = x + dense(gelu(dense(h, lp["fc1"])), lp["fc2"])
        return x, (ck, cv)

    x, new_cache = lax.scan(block, x, (p["layers"], cache[0], cache[1], xkv[0], xkv[1]))
    x = layer_norm(x, p["ln"])[:, 0]
    logits = jnp.einsum("bd,vd->bv", x, p["embed"], preferred_element_type=jnp.float32)
    return logits, new_cache


def init_cache(batch, T_max):
    shape = (config["decoder_layers"], batch, T_max, H, DH)
    return jnp.zeros(shape, DTYPE), jnp.zeros(shape, DTYPE)

## 5. Prompts, token suppression and greedy decoding

The decoder starts from a forced prompt:

```
multilingual:  <|startoftranscript|> <|lang|> <|transcribe|/|translate|> <|notimestamps|>
English-only:  <|startoftranscript|> <|notimestamps|>
```

At each step we mask:
* `suppress_tokens` from the generation config (always),
* `begin_suppress_tokens` (blank and EOS) on the first generated token,
* all special and timestamp tokens other than `<|endoftext|>`.

The whole generation loop is one `lax.while_loop` inside `jit`. It writes tokens into a fixed `[B, T_max]` buffer and stops early once **every** sequence in the batch has emitted EOS. **Language detection** is a single decoder step after `<|startoftranscript|>`, taking the argmax over the language tokens.

In [8]:
VOCAB = config["vocab_size"]
EOS = gen_cfg["eos_token_id"]
SOT = gen_cfg["decoder_start_token_id"]
NO_TS = gen_cfg["no_timestamps_token_id"]
MULTILINGUAL = gen_cfg.get("is_multilingual", "lang_to_id" in gen_cfg)
LANG_TO_ID = gen_cfg.get("lang_to_id", {})
TASK_TO_ID = gen_cfg.get("task_to_id", {})
ID_TO_LANG = {v: k[2:-2] for k, v in LANG_TO_ID.items()}
LANG_IDS = jnp.array(sorted(LANG_TO_ID.values()), jnp.int32) if MULTILINGUAL else None

suppress = np.zeros(VOCAB, bool)
suppress[[t for t in gen_cfg.get("suppress_tokens", []) if t < VOCAB]] = True
suppress[EOS + 1:] = True                       # all special/timestamp tokens (EOS itself stays allowed)
begin_suppress = np.zeros(VOCAB, bool)
begin_suppress[gen_cfg.get("begin_suppress_tokens", [])] = True
SUPPRESS_BIAS = jnp.where(jnp.asarray(suppress), -jnp.inf, 0.0)
BEGIN_BIAS = jnp.where(jnp.asarray(begin_suppress), -jnp.inf, 0.0)


def build_prompts(langs, task="transcribe"):
    if not MULTILINGUAL:
        return np.array([[SOT, NO_TS]] * len(langs), np.int32)
    return np.array([[SOT, LANG_TO_ID[f"<|{l}|>"], TASK_TO_ID[task], NO_TS] for l in langs], np.int32)


@jax.jit
def detect_language(params, xkv):
    B = xkv[0].shape[1]
    logits, _ = decoder_step(params, jnp.full((B,), SOT, jnp.int32), 0, init_cache(B, 1), xkv)
    lang_logits = logits[:, LANG_IDS]
    return LANG_IDS[jnp.argmax(lang_logits, -1)], jax.nn.softmax(lang_logits, -1).max(-1)


@functools.partial(jax.jit, static_argnames=("max_new_tokens",))
def greedy_decode(params, xkv, prompt, max_new_tokens=224):
    B, P_len = prompt.shape
    T_max = min(P_len + max_new_tokens, config["max_target_positions"])
    tokens = jnp.full((B, T_max), EOS, jnp.int32).at[:, :P_len].set(prompt)
    tokens = lax.with_sharding_constraint(tokens, BATCH)

    def cond(state):
        t, _, _, done = state
        return (t < T_max - 1) & ~jnp.all(done)

    def body(state):
        t, tokens, cache, done = state
        tok = lax.dynamic_index_in_dim(tokens, t, axis=1, keepdims=False)
        logits, cache = decoder_step(params, tok, t, cache, xkv)
        logits = logits + SUPPRESS_BIAS + jnp.where(t + 1 == P_len, BEGIN_BIAS, 0.0)
        nxt = jnp.argmax(logits, -1).astype(jnp.int32)
        forced = t + 1 < P_len
        prompt_tok = lax.dynamic_index_in_dim(tokens, jnp.minimum(t + 1, T_max - 1), axis=1, keepdims=False)
        nxt = jnp.where(forced, prompt_tok, jnp.where(done, EOS, nxt))
        tokens = lax.dynamic_update_slice_in_dim(tokens, nxt[:, None], t + 1, axis=1)
        done = done | ((nxt == EOS) & ~forced)
        return t + 1, tokens, cache, done

    init = (jnp.int32(0), tokens, init_cache(B, T_max), jnp.zeros((B,), bool))
    t_end, tokens, _, _ = lax.while_loop(cond, body, init)
    return tokens, t_end

## 6. End-to-end `transcribe()`, batched and sharded across TPU cores

The batch is padded up to a multiple of `jax.device_count()` and placed with `NamedSharding(mesh, P('data'))`, so every core processes its own slice: features, encoder and decoder. Batch sizes are also bucketed to powers of two, which keeps the number of compiled shapes small.

In [9]:
def _bucket(n):
    n = max(n, jax.device_count())
    b = 1 << (n - 1).bit_length()
    return int(math.ceil(b / jax.device_count()) * jax.device_count())


def transcribe(audios, language=None, task="transcribe", max_new_tokens=224, return_timing=False):
    # audios: list of 1-D float32 arrays at 16 kHz (each <= 30 s). language=None -> auto-detect.
    n = len(audios)
    B = _bucket(n)
    batch = np.stack([pad_or_trim(a) for a in audios] + [np.zeros(N_SAMPLES, np.float32)] * (B - n))
    timing = {}

    t0 = time.perf_counter()
    mel = log_mel_spectrogram(jax.device_put(batch, BATCH))
    enc = encode(params, mel)
    xkv = cross_kv(params, enc)
    jax.block_until_ready(xkv); timing["encode_s"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    if MULTILINGUAL and language is None:
        lang_ids, lang_probs = detect_language(params, xkv)
        langs = [ID_TO_LANG[int(i)] for i in np.asarray(lang_ids)]
    else:
        langs = [language or "en"] * B
    prompt = jax.device_put(build_prompts(langs, task), BATCH)
    tokens, steps = greedy_decode(params, xkv, prompt, max_new_tokens=max_new_tokens)
    tokens = np.asarray(tokens); timing["decode_s"] = time.perf_counter() - t0
    timing["decode_steps"] = int(steps)

    texts = [tokenizer.decode(row[row != EOS], skip_special_tokens=True).strip() for row in tokens[:n]]
    out = [{"text": t, "language": l} for t, l in zip(texts, langs[:n])]
    return (out, timing) if return_timing else out


t0 = time.time()
result = transcribe([samples[0][0]])        # first call compiles
print(f"(first call incl. compilation: {time.time() - t0:.1f}s)")
print("reference :", samples[0][1])
print("whisper   :", result[0])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


(first call incl. compilation: 10.7s)
reference : MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
whisper   : {'text': 'Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.', 'language': 'en'}


## 7. Accuracy: WER on LibriSpeech samples

We transcribe all 73 utterances in batches and compute word error rate after Whisper's English text normalizer (so `Mr.` and `MISTER` count as the same word). Published LibriSpeech test-clean WERs for reference: tiny ≈ 7.5 %, base ≈ 5 %, small ≈ 3.4 %, large-v3 ≈ 2 %.

In [10]:
def normalize(text):
    try:
        return tokenizer.normalize(text)
    except Exception:
        return " ".join(re.sub(r"[^a-z0-9' ]", " ", text.lower()).split())

def word_errors(ref, hyp):
    r, h = ref.split(), hyp.split()
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
            prev, d[j] = d[j], cur
    return d[len(h)], len(r)

EVAL_BATCH = 32 if ON_TPU else 16
hyps, t0 = [], time.time()
for i in range(0, len(samples), EVAL_BATCH):
    chunk = samples[i:i + EVAL_BATCH]
    hyps += [r["text"] for r in transcribe([w for w, _ in chunk], language="en")]
elapsed = time.time() - t0

errs = [word_errors(normalize(ref), normalize(hyp)) for (_, ref), hyp in zip(samples, hyps)]
wer = sum(e for e, _ in errs) / sum(n for _, n in errs)
print(f"WER = {100 * wer:.2f}%  over {len(samples)} utterances ({total_sec:.0f}s audio) in {elapsed:.1f}s "
      f"(includes compiling new batch shapes)")
for (_, ref), hyp in list(zip(samples, hyps))[:5]:
    print(f"\n  REF: {ref}\n  HYP: {hyp}")

WER = 3.17%  over 73 utterances (481s audio) in 44.9s (includes compiling new batch shapes)

  REF: MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
  HYP: Mr. Quilter is the apostle of the middle classes, and we are glad to welcome his gospel.

  REF: NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
  HYP: Nor is Mr. Quilter's manner less interesting than his matter.

  REF: HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND
  HYP: He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similes drawn from eating and its results occur most readily to the mind.

  REF: HE HAS GRAVE DOUBTS WHETHER SIR FREDERICK LEIGHTON'S WORK IS REALLY GREEK AFTER ALL AND CAN DISCOVER IN IT BUT LITTLE OF ROCKY ITHACA
  HYP: He has grave doubts whether Sir Frederick Layton's work is r

## 8. Long-form audio (> 30 s)

Whisper sees at most 30 s at a time. A simple, fast strategy that suits the TPU well:

1. Split the recording into chunks of ≤30 s, cutting each one at the **quietest 100 ms window** in its last 5 seconds so words aren't cut in half.
2. Transcribe **all chunks in one batch** (in parallel across TPU cores).
3. Join the texts.

As a demo we join all the LibriSpeech utterances into one ~6-minute recording.

In [11]:
def split_on_silence(audio, max_len=N_SAMPLES, search=5 * SAMPLE_RATE, win=1600):
    chunks, start = [], 0
    while len(audio) - start > max_len:
        lo, hi = start + max_len - search, start + max_len
        seg = audio[lo:hi]
        energy = np.convolve(seg ** 2, np.ones(win), mode="valid")
        cut = lo + int(np.argmin(energy)) + win // 2
        chunks.append(audio[start:cut]); start = cut
    chunks.append(audio[start:])
    return chunks


def transcribe_long(audio, batch_size=32, **kw):
    chunks = split_on_silence(audio)
    texts = []
    for i in range(0, len(chunks), batch_size):
        texts += [r["text"] for r in transcribe(chunks[i:i + batch_size], **kw)]
    return " ".join(texts), len(chunks)


long_audio = np.concatenate([w for w, _ in samples])
t0 = time.time()
text, n_chunks = transcribe_long(long_audio, language="en")
dt = time.time() - t0
print(f"{len(long_audio)/SAMPLE_RATE/60:.1f} min of audio -> {n_chunks} chunks, transcribed in {dt:.1f}s "
      f"({len(long_audio)/SAMPLE_RATE/dt:.0f}x real time, incl. any compilation)\n")
print(text[:1500], "...")

8.0 min of audio -> 18 chunks, transcribed in 1.8s (271x real time, incl. any compilation)

Mr. Quilter is the apostle of the middle classes and we are glad to welcome his gospel. Nor is Mr. Quilter's manner less interesting than his matter. He tells us that at this festive season of the year, with Christmas and roast beef looming before us, similes drawn from eating and its results occur most readily to the mind. He has grave doubts whether Sir Frederick Layton's work is really Greek after all, and can discover in it but little of rocky Ithaca. Linnell's pictures are a sort of Up Guards and Adam paintings, and Mason's exquisite idylls are as national as a jingo poem. Mr. Burkett Foster's landscapes smile at one much in the same way that Mr. Carker used to flash his teeth. Mr. John Collier gives his sitter a cheerful slap on the back before he says like a shampooer in a Turkish bath next man It is obviously unnecessary for us to point out how luminous these criticisms are how delicate 

## 9. Language detection and translation

Whisper detects the spoken language itself (multilingual checkpoints only). `task="translate"` produces English text from any supported language. (`large-v3-turbo` was not trained for translation, so use `large-v3`, `medium` or `small` for that.)

In [12]:
if MULTILINGUAL:
    res = transcribe([samples[1][0], samples[2][0]])            # language=None -> auto-detect
    for r in res:
        print(f"[{r['language']}] {r['text']}")
else:
    print("English-only checkpoint: no language detection.")

[en] Nor is Mr. Quilter's manner less interesting than his matter.
[en] he tells us that at this festive season of the year with christmas and roast beef looming before us similes drawn from eating and its results occur most readily to the mind


### Try your own audio (optional)

Upload any audio file (wav, mp3, flac, m4a…). It is resampled to 16 kHz with `librosa`, and the language is auto-detected.

In [13]:
TRY_UPLOAD = False   # set True to upload a file in Colab

if TRY_UPLOAD:
    import librosa
    from google.colab import files
    for name in files.upload():
        audio, _ = librosa.load(name, sr=SAMPLE_RATE, mono=True)
        if len(audio) <= N_SAMPLES:
            print(name, "->", transcribe([audio]))
        else:
            print(name, "->", transcribe_long(audio)[0])

## 10. Benchmark: real-time factor vs batch size

Each call has 30 s chunks of real speech. We report **encoder time**, **decoder time**, and **throughput as a multiple of real time** (seconds of audio transcribed per second). The first call at each batch size compiles, so it is excluded.

In [14]:
BATCH_SIZES = [1, 8, 16, 32, 64] if ON_TPU else [1, 4, 8]
bench_chunks = split_on_silence(long_audio)
print(f"{MODEL_ID} | {jnp.dtype(DTYPE).name} | {jax.device_count()} device(s)\n")
print(f"{'batch':>5} | {'encode ms':>9} | {'decode ms':>9} | {'steps':>5} | {'total ms':>8} | {'x real-time':>11}")
for bs in BATCH_SIZES:
    audios = [bench_chunks[i % len(bench_chunks)] for i in range(bs)]
    transcribe(audios, language="en")                          # compile / warm-up
    reps = 3
    enc_t = dec_t = 0.0
    for _ in range(reps):
        _, tm = transcribe(audios, language="en", return_timing=True)
        enc_t += tm["encode_s"] / reps; dec_t += tm["decode_s"] / reps
    audio_sec = sum(min(len(a), N_SAMPLES) for a in audios) / SAMPLE_RATE
    total = enc_t + dec_t
    print(f"{bs:>5} | {enc_t*1e3:9.1f} | {dec_t*1e3:9.1f} | {tm['decode_steps']:>5} | {total*1e3:8.1f} | {audio_sec/total:10.0f}x")

openai/whisper-large-v3-turbo | bfloat16 | 1 device(s)

batch | encode ms | decode ms | steps | total ms | x real-time
    1 |      35.2 |      52.9 |    90 |     88.1 |        311x
    8 |     285.3 |     138.8 |   117 |    424.1 |        522x
   16 |     542.8 |     251.8 |   117 |    794.6 |        558x
   32 |    1047.0 |     453.1 |   117 |   1500.1 |        579x
   64 |    1640.2 |     983.0 |   117 |   2623.2 |        655x
